In [ ]:
from datetime import datetime

import pandas as pd
import numpy as np
from scipy import stats

### 1.  Выбор данных, полученных во время эксперимента

In [ ]:
def get_data_subset(df, begin_date=None, end_date=None, user_ids=None, columns=None):
    """Возвращает подмножество данных.

    :param df (pd.DataFrame): таблица с данными, обязательные столбцы: 'date', 'user_id'.
    :param begin_date (datetime.datetime | None): дата начала интервала с данными.
        Пример, df[df['date'] >= begin_date].
        Если None, то фильтровать не нужно.
    :param end_date (datetime.datetime | None): дата окончания интервала с данными.
        Пример, df[df['date'] < end_date].
        Если None, то фильтровать не нужно.
    :param user_ids (list[str] | None): список user_id, по которым нужно предоставить данные.
        Пример, df[df['user_id'].isin(user_ids)].
        Если None, то фильтровать по user_id не нужно.
    :param columns (list[str] | None): список названий столбцов, по которым нужно предоставить данные.
        Пример, df[columns].
        Если None, то фильтровать по columns не нужно.

    :return df (pd.DataFrame): датафрейм с подмножеством данных.
    """
    if end_date is None:
        end_date = datetime.now()
    if begin_date is None:
        begin_date = df.date.min()
    if user_ids and columns:
        return df[(df.date >= begin_date) & (df.date < end_date) & (df.user_id.isin(user_ids))][columns]
    elif user_ids:
        return df[(df.date >= begin_date) & (df.date < end_date) & (df.user_id.isin(user_ids))]
    elif columns:
        return df[(df.date >= begin_date) & (df.date < end_date)][columns]
    else:
        return df[(df.date >= begin_date) & (df.date < end_date)]

In [ ]:
df = pd.DataFrame({
    'date': [datetime(2022, 1, 5), datetime(2022, 1, 7)],
    'user_id': ['1', '2'],
})
new_df = get_data_subset(df, datetime(2022, 1, 1), datetime(2022, 1, 6))

### 2. Время обработки запроса сервером

In [ ]:
def get_response_time(df_web_logs, begin_date, end_date):
    """Вычисляет значения времени обработки запроса сервером.

    Нужно вернуть значения user_id и load_time из таблицы df_web_logs,
    отфильтрованные по дате.
    Считаем, что запросы обрабатываются независимо, поэтому группировать
    по user_id не нужно.

    :param df_web_logs (pd.DataFrame): таблица с логами сайта, содержит
    столбцы ['user_id', 'date', 'load_time'].
    :param begin_date, end_date (datetime): границы периода для
    фильтрации данных по дате. Левая граница входит, правая не входит.

    :return (pd.DataFrame): датафрейм с двумя столбцами ['user_id', 'metric']
    """
    if end_date is None:
        end_date = datetime.now()
    if begin_date is None:
        begin_date = df_web_logs.date.min()
    result = df_web_logs[(df_web_logs.date >= begin_date) & (df_web_logs.date < end_date)][['user_id', 'load_time']]
    result.columns= ['user_id', 'metric']
    return result

### 3. Выручка с пользователя за указанный период

In [ ]:
def get_revenue_web(df_sales, df_web_logs, begin_date, end_date):
    """Вычисляет значения выручки с пользователя за указанный период
    для заходивших на сайт в указанный период.

    Эти данные нужны для экспериментов на сайте, когда в эксперимент
    попадают только те, кто заходил на сайт во время эксперимента.

    Нужно вернуть значения user_id и выручки (sum(price)) за указанный
    период для пользователей, заходивших на сайт в указанный период.
    Если пользователь зашёл на сайт и ничего не купил, его суммарная
    стоимость покупок равна нулю.
    Для каждого user_id должно быть ровно одно значение.

    :param df_sales (pd.DataFrame): таблица с продажами, содержит
        столбцы ['user_id', 'date', 'price'].
    :param df_web_logs (pd.DataFrame): таблица с логами сайта, содержит
        столбцы ['user_id', 'date', 'load_time'].
    :param begin_date, end_date (datetime): границы периода для фильтрации
        данных по дате. Левая граница входит, правая не входит.

    :return (pd.DataFrame): датафрейм с двумя столбцами ['user_id', 'metric']
    """
    # handle cases with None initial parameters
    if end_date is None:
        end_date = datetime.now()
    if begin_date is None:
        begin_date = df_web_logs.date.min()
    # get users, visited site in period under investigation
    users = df_web_logs[(df_web_logs.date >= begin_date) & (df_web_logs.date < end_date)][['user_id']].drop_duplicates()
    # get sales in period under investigation
    sales = df_sales[
                        (df_sales.date >= begin_date) 
                        & (df_sales.date < end_date) 
                        & (df_sales['user_id'].isin(users.user_id.to_list()))
                    ][['user_id', 'price']]
    sales = sales.groupby(['user_id'], as_index=False).agg(metric = ('price', 'sum'))
    # merge dataframes
    return users.merge(sales, on=['user_id'], how='left').fillna(0)

### 4. Значения выручки с пользователя за указанный период для заходивших на сайт до end_date

In [ ]:
def get_revenue_all(df_sales, df_web_logs, begin_date, end_date):
    """Вычисляет значения выручки с пользователя за указанный период
    для заходивших на сайт до end_date.

    Эти данные нужны, например, для экспериментов с рассылкой по email,
    когда в эксперимент попадают те, кто когда-либо оставил нам свои данные.

    Нужно вернуть значения user_id и выручки (sum(price)) за указанный период
    для пользователей, заходивших на сайт до end_date.
    Если пользователь ничего не купил за указанный период, его суммарная
    стоимость покупок равна нулю.
    Для каждого user_id должно быть ровно одно значение.

    :param df_sales (pd.DataFrame): таблица с продажами, содержит
        столбцы ['user_id', 'date', 'price'].
    :param df_web_logs (pd.DataFrame): таблица с логами сайта, содержит
        столбцы ['user_id', 'date', 'load_time'].
    :param begin_date, end_date (datetime): границы периода для фильтрации
        данных по дате. Левая граница входит, правая не входит.

    :return (pd.DataFrame): датафрейм с двумя столбцами ['user_id', 'metric']
    """
    # handle cases with None initial parameters
    if end_date is None:
        end_date = datetime.now()
    if begin_date is None:
        begin_date = df_web_logs.date.min()
    # get users, visited site before end_date
    users = df_web_logs[(df_web_logs.date < end_date)][['user_id']].drop_duplicates()
    # get sales in period under investigation
    sales = df_sales[
                        (df_sales.date >= begin_date) 
                        & (df_sales.date < end_date) 
                        & (df_sales['user_id'].isin(users.user_id.to_list()))
                    ][['user_id', 'price']]
    sales = sales.groupby(['user_id'], as_index=False).agg(metric = ('price', 'sum'))
    # merge dataframes
    return users.merge(sales, on=['user_id'], how='left').fillna(0)

### 5. Расчет p-value

In [ ]:
def get_ttest_pvalue(metrics_a_group, metrics_b_group):
    """Применяет тест Стьюдента, возвращает pvalue.

    :param metrics_a_group (np.array): массив значений метрик группы A
    :param metrics_a_group (np.array): массив значений метрик группы B
    :return (float): значение p-value
    """
    return stats.ttest_ind(metrics_a_group, metrics_b_group).pvalue.item()

### 6. Расчет необходимого размера выборки

MDE из доступного объема /2 (если 2 варианта)

In [ ]:
def get_minimal_determinable_effect(std, sample_size, alpha=0.05, beta=0.2):
    t_alpha = norm.ppf(1 - alpha / 2, loc=0, scale=1)
    t_beta = norm.ppf(1 - beta, loc=0, scale=1)
    disp_sum_sqrt = (2 * (std ** 2)) ** 0.5
    mde = (t_alpha + t_beta) * disp_sum_sqrt / np.sqrt(sample_size)
    return mde

Расчет размера выборки